## Load data

In [3]:
import os, sys
os.environ['CHIMERA_USE_x64'] = "False"
os.environ['CHIMERA_ENABLE_GPU'] = "True"
sys.path.append(os.getcwd()+'/../')
from CHIMERA import data as data
import jax.numpy as jnp 

dir_data = './data/'
file_ev = dir_data+'GWTC5_242CBC_FAR0.25_PE2048.h5'
theta_pe_det = data.load_gw_pe_samples(file_ev, 
                                 parameters=['m1det', 'm2det', 'dL'], 
                                 return_struct=True)

pe_prior = jnp.load(f'{dir_data}GWTC5_242CBC_FAR0.25_PE2048prior.npy')
theta_pe_det = theta_pe_det.update(pe_prior=pe_prior)
nev = len(theta_pe_det.dL)
print(f'Loaded {nev} GW events')
a = jnp.array([1.0])
a.dtype

Loaded 242 GW events


dtype('float32')

## Instantiate population model

In [4]:
from CHIMERA.cosmo import flrw
from CHIMERA.mass.paired import bpl_dip_three_peaks
from CHIMERA.rate import madau_dickinson
from CHIMERA import population

# define population models with some fiducials
cosmo = flrw(H0 = 70., Om0=0.3, z_max = 5.)
mass = bpl_dip_three_peaks(m_low = 0.8, m_high = 165.)
rate = madau_dickinson(gamma = 2.2, kappa =  3., zp = 2.)

population = population(cosmo, mass, rate, scale_free=True)

## Instantiate the Selection Function module

In [5]:
from CHIMERA import selection_function

file_inj = dir_data + "GWTC5_injections_far0.25.h5"
theta_inj_det = data.load_injection_data(file_inj, key_mapping={'snr':'snr', 'log_pdraw':'log_pdraw'}, frame='detector')
sel_fcn = selection_function(theta_inj_det, N_inj = 1568035640.0) 

## Instantiate the Hyperlike

In [6]:
from CHIMERA import hyperlikelihood
import jax

hyperlike = hyperlikelihood(  # data
  theta_gw_det = theta_pe_det,
  # population 
  population=population,
  # integration grid resolution
  z_grids_res = 300,  
  # selection function
  selection_function=sel_fcn,
  # numerical stability
  pe_neff = 2.0,
  inj_neff = None, # default to 5*Nev  
  # KDE settings
  kind_kde = 'fft',
  kernel = 'gaussian',
  kde_bw = None, # default to scott
  num_bins = 200,
)


2026-09-10 16:39:00,456 - CHIMERA - WARNING - `kde_bw` is None, using Scott rule for bandwidth.
2026-09-10 16:39:00,456 - CHIMERA - INFO - Created hyperlikelihood model. Using 242 GW events.


## gradient check

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import h5py
from CHIMERA.utils.config import USE_x64

@jax.jit(static_argnames=['param'])
def loglike_fn(value, param):
    return hyperlike(**{param: value})

@jax.jit(static_argnames=['param'])
def loglike_and_grad_fn(value, param):
    func = lambda val: loglike_fn(val, param)
    return jax.value_and_grad(func)(value)

# Priors: full parameter set of `bpl_dip_three_peaks` (mass) + `madau_dickinson` (rate) + H0
priors = {"H0": [10.,200.],
  "alpha_1": [-4,12.],
  "alpha_2": [-4,12.],
  "beta_bottom": [-4.,12.],
  "beta_top": [-4.,12.],
  "mu_g1": [5.,150.],       
  "sigma_g1": [0.4,10.],               
  "mu_g2":[5.0, 150.],
  "sigma_g2":[0.4, 15.],
  "mu_g3":[5.0, 150.],
  "sigma_g3":[0.4, 15.],
  "lambda_g":[0,1],
  "lambda_1": [0,1],
  "lambda_2":[0,1],
  "bottomsmooth": [0.01,1.],
  "topsmooth": [0.001,1.],
  "leftdip": [1.5,3.0],
  "rightdip": [5.0, 9.0],
  "leftdipsmooth": [0.01,2.0],
  "rightdipsmooth": [0.01,2.0],
  "deep": [0., 1.],  
  "m_low": [0.4,1.4],
  "m_high": [50.,200.],
  "gamma": [0.,12.],
  "kappa": [0.,6.],
  "zp": [0.,4.]
}

trues = {'H0':70., **population.mass.as_dict, **population.rate.as_dict}
assert set(priors) == set(trues), f"priors/trues mismatch: {set(priors) ^ set(trues)}"

n_grid = 99
out_file = f"results_{'float64' if USE_x64 else 'float32'}.h5"

nan_summary = {}
with h5py.File(out_file, 'w') as f:
    f.attrs['trues'] = jnp.array(list(trues.values()))
    f.attrs['par_names'] = np.array(list(trues.keys()), dtype='S') 
    
    for par, bounds in priors.items():
        grp = f.create_group(par)
        x = jnp.linspace(bounds[0], bounds[1], n_grid)
        x = jnp.sort(jnp.concatenate([x, jnp.array([trues[par]])]))
        grp.create_dataset('x', data=x)
        
        loglike = jnp.empty(x.shape[0])
        grad = jnp.empty(x.shape[0])
        for i, val in enumerate(tqdm(x, desc=par)):
            ll, g = loglike_and_grad_fn(val, par)
            loglike = loglike.at[i].set(ll)
            grad = grad.at[i].set(g)
        
        n_nan_like = int(jnp.isnan(loglike).sum())
        n_nan_grad = int(jnp.isnan(grad).sum())
        n_inf_grad = int(jnp.isinf(grad).sum())
        nan_summary[par] = (n_nan_like, n_nan_grad, n_inf_grad)
        if n_nan_like or n_nan_grad or n_inf_grad:
            print(f"[gradient check] {par}: {n_nan_like}/{x.shape[0]} NaN loglike, "
                  f"{n_nan_grad} NaN grad, {n_inf_grad} inf grad")
        
        loglike -= jnp.nanmax(loglike)
        post = jnp.exp(loglike)
        post /= jnp.trapezoid(post, x=x)
        
        grp.create_dataset('post', data=post)
        grp.create_dataset('grad', data=grad)

        # Free the per-`param` compiled XLA executable: static_argnames=['param']
        # makes jax.jit cache a *separate* program for every distinct param name,
        # and on a small GPU (e.g. 4GB) this accumulates until CUDA OOMs partway
        # through the sweep. Clearing here keeps only one compiled program alive
        # at a time.
        jax.clear_caches()

print(f"\nSaved gradient-check results to '{out_file}' (USE_x64={USE_x64})")
bad_pars = [p for p, (nl, ng, ni) in nan_summary.items() if nl or ng or ni]
if bad_pars:
    print(f"Parameters with NaN/inf loglike or gradient: {bad_pars}")
else:
    print("No NaN/inf found in loglike or gradient for any parameter.")

alpha_1:  15%|█████▎                             | 15/100 [00:06<00:21,  3.99it/s]

In [ ]:
# Plot from loaded results
with h5py.File('results_float64.h5', 'r') as f64, h5py.File('results_float32.h5', 'r') as f32:
    trues_dict = dict(zip(
        [name.decode('utf-8') for name in f64.attrs['par_names']], 
        f64.attrs['trues']
    ))
    for par in priors:
        grp = f64[par]
        x = grp['x'][:]
        post64 = grp['post'][:]
        grad64 = grp['grad'][:]
        print(par, '64', np.isnan(grad64).sum())

        grp = f32[par]
        post32 = grp['post'][:]
        grad32 = grp['grad'][:]
        print(par, '32', np.isnan(grad32).sum())
        
        fig, axs = plt.subplots(1, 2, figsize=(10,4))
        axs[0].plot(x, post64)
        axs[1].plot(x, grad64)
        
        axs[0].plot(x, post32)
        axs[1].plot(x, grad32)

        for ax in axs:
            ax.set_xlabel(par)
            ax.axvline(trues_dict[par], c='k', ls='--')
        axs[1].axhline(0., c='k', ls='--')
        plt.show()

In [ ]:
# Plot from loaded results
with h5py.File('results_float64.h5', 'r') as f64, h5py.File('results_float32.h5', 'r') as f32:
    trues_dict = dict(zip(
        [name.decode('utf-8') for name in f64.attrs['par_names']], 
        f64.attrs['trues']
    ))
    for par in priors:
        grp = f64[par]
        x = grp['x'][:]
        post64 = grp['post'][:]
        grad64 = grp['grad'][:]

        grp = f32[par]
        post32 = grp['post'][:]
        grad32 = grp['grad'][:]
        
        fig, axs = plt.subplots(1, 2, figsize=(10,4))
        axs[0].plot(x, post64 - post32, label = 'post64 - post32')
        axs[1].plot(x, grad64 - grad32, label = 'grad64 - grad32')
        for ax in axs:
            ax.legend()
            ax.set_xlabel(par)
            ax.axhline(0., c='k', ls='--')
        plt.savefig(f'diff_{par}.png')
        plt.show()

In [ ]:
with h5py.File('results_float32.h5', 'r') as f32, h5py.File('results_float64.h5', 'r') as f64:
    #plt.plot(f32['H0']['x'][:], f32['H0']['grad'][:])
    plt.plot(f64['H0']['x'][:], f64['H0']['grad'][:])
    